# Nagi Studio — See-through 顔・髪・体の分割

1. ランタイムをGPUへ変更します。2160px顔はL4 24GB以上、A100推奨です（達成保証ではありません）。
2. 上から実行し、アプリの「顔ジョブ ZIP」または「髪・体ジョブ ZIP」をアップロードします。
3. 完了した `see-through-result.zip` をアプリの「See-through結果 ZIP」から読み込みます。
4. 顔の位置、不要レイヤー、保護した編集を確認し「採用」を押します。

顔結果には眉毛・まつげ・口・鼻・肌・耳・瞳・白目の8種類のみを書き出します。
実装内部の推論は公式の全レイヤー処理ですが、その他の顔レイヤーは結果ZIPから除外します。
顔の2160px入力は2176pxへ余白追加し、結果では除去します。縮小フォールバックはありません。
初回は公式コードとモデルを取得します。Drive保存を選ぶと中間結果・ログ・ZIPを残せます。


In [ ]:
# 設定
USE_DRIVE = True
GROUP_OFFLOAD = True
REPO_COMMIT = "58a1cb11d13f85acec9bbddb8cd4b6487843d4cf"
DRIVE_FOLDER = "NagiStudio-SeeThrough"


In [ ]:
import os, sys, subprocess, json, hashlib, zipfile, io
from pathlib import Path
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True).strip()
print(gpu)
if int(gpu.splitlines()[0].rsplit(",", 1)[1]) < 22000:
    raise RuntimeError("2160px処理にはL4/A100等の22GB以上のGPUを選択してください。解像度は下げません。")
from google.colab import drive, files
if USE_DRIVE:
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
else:
    OUTPUT_ROOT = Path("/content/nagi-output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# 準備。ノートブック本体でtorch/PILをimportせず、推論は新しいPythonプロセスで行います。
REPO = Path("/content/nagi-see-through")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/shitagaki-lab/see-through.git", str(REPO)], check=True)
subprocess.run(["git", "fetch", "origin", REPO_COMMIT], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", REPO_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.8.0+cu128", "torchvision==0.23.0+cu128", "torchaudio==2.8.0+cu128", "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO, check=True)
assets = REPO / "assets"
if not assets.exists():
    assets.symlink_to(REPO / "common/assets", target_is_directory=True)
subprocess.run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(); print(torch.__version__, torch.cuda.get_device_name())"], check=True)


In [ ]:
# ジョブを選択。ZIP内のPythonコードを実行することはありません。
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("ジョブZIPを1つだけ選択してください")
payload = next(iter(uploaded.values()))
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    if sum(i.file_size for i in archive.infolist()) > 100_000_000:
        raise ValueError("ジョブが大きすぎます")
    job = json.loads(archive.read("job.json"))
    if job.get("backend") != "see-through" or job.get("scope") not in ("face", "body"):
        raise ValueError("Nagi StudioのSee-throughジョブではありません")
    if not __import__("re").fullmatch(r"[a-f0-9]{32}", job["id"]):
        raise ValueError("Invalid job ID")
JOB = OUTPUT_ROOT / (job["id"] + ".zip")
JOB.write_bytes(payload)
RESULT = OUTPUT_ROOT / (job["id"] + "-result.zip")
print("Scope:", job["scope"], "Settings:", job["settings"], "Result:", RESULT)


In [ ]:
# このノートブック生成時点のワーカーを固定。ユーザーZIP由来のコードは使いません。
WORKER = Path("/content/nagi-seethrough-worker.py")
WORKER.write_text('"""Standalone Colab/local See-through runner and strict result packager."""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport zipfile\nfrom PIL import Image\n\nREPO_COMMIT = "58a1cb11d13f85acec9bbddb8cd4b6487843d4cf"\nFACE_TAGS = {"eyebrow":"眉毛", "eyelash":"まつげ", "mouth":"口", "nose":"鼻",\n             "face":"肌", "ears":"耳", "irides":"瞳", "eyewhite":"白目"}\nBODY_TAGS = {"back hair":"後ろ髪", "front hair":"前髪", "neck":"首", "topwear":"胸・上衣（袖を含む場合あり）",\n             "handwear":"腕・手（モデル推定）", "bottomwear":"スカート・下衣", "legwear":"足・脚衣", "footwear":"靴"}\n\n\ndef prepare_face(image):\n    if image.width != image.height or image.width < 2160:\n        raise ValueError("Face input must be square and at least 2160px. Upscale with FFmpeg before export.")\n    size = math.ceil(image.width/64)*64\n    offset = (size-image.width)//2\n    canvas = Image.new("RGBA", (size,size), (255,255,255,255))\n    canvas.alpha_composite(image.convert("RGBA"), (offset,offset))\n    return canvas.convert("RGB"), offset\n\n\ndef prepare_input(image, scope):\n    if scope == "face":\n        padded,offset=prepare_face(image)\n        return padded,[offset,offset]\n    size=math.ceil(max(image.size)/64)*64\n    offset=[(size-image.width)//2,(size-image.height)//2]\n    canvas=Image.new("RGBA",(size,size),(255,255,255,255))\n    canvas.alpha_composite(image.convert("RGBA"),tuple(offset))\n    return canvas.convert("RGB"),offset\n\n\ndef read_job(path):\n    with zipfile.ZipFile(path) as archive:\n        if len(archive.infolist())>30 or sum(x.file_size for x in archive.infolist())>100_000_000:\n            raise ValueError("Oversized job")\n        job=json.loads(archive.read("job.json"))\n        if job.get("schemaVersion")!=1 or job.get("backend")!="see-through" or job.get("scope") not in ("face","body"):\n            raise ValueError("Unsupported job")\n        if not re.fullmatch(r"[a-f0-9]{32}",job.get("id","")):raise ValueError("Invalid job ID")\n        if job.get("repoCommit")!=REPO_COMMIT:raise ValueError("Unsupported worker version")\n        settings=job.get("settings",{})\n        if type(settings.get("seed")) is not int or not 0<=settings["seed"]<2**32 or type(settings.get("steps")) is not int or not 1<=settings["steps"]<=100:\n            raise ValueError("Invalid inference settings")\n        source=archive.read("input.png")\n        if hashlib.sha256(source).hexdigest()!=job["inputSha256"]:\n            raise ValueError("Input hash mismatch")\n        image=Image.open(io.BytesIO(source))\n        if image.width*image.height>20_000_000:raise ValueError("Image too large")\n        image=image.convert("RGBA")\n    return job,image\n\n\ndef package_result(job, image, padded_size, offset, layer_dir, result_path, execution):\n    """Whitelist output and map padded model coordinates back to source pixels."""\n    tags=FACE_TAGS if job["scope"]=="face" else BODY_TAGS\n    result={"schemaVersion":1,"jobId":job["id"],"projectId":job["projectId"],\n            "inputSha256":job["inputSha256"],"scope":job["scope"],"status":"success",\n            "backend":"see-through","sourceSize":list(image.size),"inferenceResolution":padded_size,\n            "paddingOffset":offset,"layers":[],"missing":[],"execution":execution}\n    destination=Path(result_path);temp=destination.with_suffix(".tmp")\n    with zipfile.ZipFile(temp,"w",zipfile.ZIP_DEFLATED) as archive:\n        for tag,label in tags.items():\n            path=Path(layer_dir)/f"{tag}.png"\n            if not path.is_file():\n                result["missing"].append(tag);continue\n            layer=Image.open(path).convert("RGBA")\n            if layer.size!=(padded_size,padded_size):\n                raise ValueError(f"Unexpected {tag} dimensions {layer.size}; no silent resizing")\n            layer=layer.crop((offset[0],offset[1],offset[0]+image.width,offset[1]+image.height))\n            if not layer.getchannel("A").getbbox():\n                result["missing"].append(tag);continue\n            filename=f"layers/{tag.replace(\' \',\'_\')}.png"\n            output=io.BytesIO();layer.save(output,"PNG");payload=output.getvalue()\n            archive.writestr(filename,payload)\n            result["layers"].append({"tag":tag,"name":label,"image":filename,"sha256":hashlib.sha256(payload).hexdigest()})\n        if not result["layers"]:raise ValueError("No requested nonempty layers were generated")\n        archive.writestr("result.json",json.dumps(result,ensure_ascii=False))\n    os.replace(temp,destination)\n    return result\n\n\ndef run(job_path,result_path,repo,group_offload=False,work_dir=None):\n    job,image=read_job(job_path);repo=Path(repo).resolve()\n    script=repo/"inference/scripts/inference_psd.py"\n    if not script.is_file():raise ValueError("Choose an official See-through checkout")\n    revision=subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip()\n    if revision!=REPO_COMMIT:raise ValueError(f"See-through checkout must be pinned to {REPO_COMMIT}")\n    work=Path(work_dir or Path(result_path).with_suffix(".work")).resolve()/job["id"]\n    work.mkdir(parents=True,exist_ok=True)\n    padded,offset=prepare_input(image,job["scope"])\n    input_path=work/"input.png";padded.save(input_path)\n    output=work/"output";marker=work/"inference-complete.json"\n    parameters={"inputSha256":job["inputSha256"],"resolution":padded.width,\n                "seed":job["settings"]["seed"],"steps":job["settings"]["steps"],"commit":revision}\n    command=[sys.executable,str(script),"--srcp",str(input_path),"--save_dir",str(output),\n             "--resolution",str(padded.width),"--resolution_depth","768",\n             "--inference_steps",str(parameters["steps"]),"--seed",str(parameters["seed"]),"--save_to_psd"]\n    if group_offload:command.append("--group_offload")\n    start=time.monotonic()\n    if not marker.exists() or json.loads(marker.read_text("utf-8"))!=parameters:\n        print("Running:"," ".join(command),flush=True)\n        env=os.environ.copy();env["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"\n        with (work/"inference.log").open("w",encoding="utf-8") as log:\n            process=subprocess.Popen(command,cwd=repo,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding="utf-8",errors="replace")\n            for line in process.stdout:\n                print(line,end="",flush=True);log.write(line);log.flush()\n            code=process.wait()\n        if code:\n            (work/"failure.json").write_text(json.dumps({"status":"failed","jobId":job["id"],"exitCode":code}),"utf-8")\n            raise RuntimeError(f"See-through failed ({code}). Log: {work/\'inference.log\'}. Resolution was not reduced.")\n        marker.write_text(json.dumps(parameters),"utf-8")\n    import torch\n    execution={**parameters,"torch":torch.__version__,"gpu":torch.cuda.get_device_name(),\n               "secondsThisRun":time.monotonic()-start,"groupOffload":group_offload,\n               "model":"layerdifforg/seethroughv0.0.2_layerdiff3d","inferred":True}\n    result=package_result(job,image,padded.width,offset,output/"input",result_path,execution)\n    print(json.dumps(result,ensure_ascii=False,indent=2))\n\n\nif __name__=="__main__":\n    parser=argparse.ArgumentParser()\n    parser.add_argument("job");parser.add_argument("result");parser.add_argument("--repo",required=True)\n    parser.add_argument("--group-offload",action="store_true");parser.add_argument("--work-dir")\n    args=parser.parse_args()\n    run(args.job,args.result,args.repo,args.group_offload,args.work_dir)\n', encoding="utf-8")
command = [sys.executable, str(WORKER), str(JOB), str(RESULT), "--repo", str(REPO), "--work-dir", str(OUTPUT_ROOT / "work")]
if GROUP_OFFLOAD:
    command.append("--group-offload")
subprocess.run(command, check=True)


In [ ]:
# 成功した結果をローカルへダウンロード
with zipfile.ZipFile(RESULT) as archive:
    result = json.loads(archive.read("result.json"))
assert result["status"] == "success"
print("抽出:", [item["name"] for item in result["layers"]])
print("未検出:", result["missing"])
files.download(str(RESULT))
